<a href="https://colab.research.google.com/github/Ritesh0912-coder/emily-Llm-model-/blob/main/train_emily_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ══════════════════════════════════════════════════
#  EMILY FULL RESTART — Run this after every reset
# ══════════════════════════════════════════════════
import os

# Step 1: Clone project from GitHub
print("📥 Cloning project from GitHub...", flush=True)
os.chdir('/content')
!git clone https://github.com/Ritesh0912-coder/emily-Llm-model-.git project
os.chdir('/content/project')

# Step 2: Install dependencies
print("\n📦 Installing dependencies...", flush=True)
!pip install -q fastapi uvicorn pydantic pyyaml tokenizers tqdm

# Step 3: Apply all necessary fixes
print("\n🔧 Applying fixes...", flush=True)

# Fix 1 — dedup=False in bootstrap.py
with open("bootstrap.py", "r") as f:
    code = f.read()
code = code.replace("dedup=True", "dedup=False")
with open("bootstrap.py", "w") as f:
    f.write(code)

# Fix 2 — write correct pro_gpu config
config = """
model:
  name: emily-pro
  vocab_size: 4096
  context_length: 128
  d_model: 512
  n_heads: 8
  n_kv_heads: 8
  n_layers: 8
  d_ff: 2048
  dropout: 0.1
  attention_type: standard
  tie_embeddings: true
  use_rms_norm: true
  use_swiglu: true
  use_rope: true
  rope_base: 10000
  max_seq_len: 128

tokenizer:
  model_type: bpe
  vocab_size: 4096
  model_path: checkpoints/emily-pro/tokenizer.json

training:
  batch_size: 8
  gradient_accumulation_steps: 4
  max_steps: 10000
  eval_interval: 500
  save_interval: 1000
  log_interval: 50
  learning_rate: 3.0e-4
  min_lr: 3.0e-5
  warmup_steps: 500
  weight_decay: 0.1
  max_grad_norm: 1.0
  optimizer: adamw
  scheduler: cosine
  use_amp: true
  amp_dtype: float16
  seed: 42
  compile: false
  checkpoint_dir: checkpoints/emily-pro
  log_dir: logs/emily-pro
  wandb_enabled: false

dataset:
  train_path: datasets/tokenized/train.bin
  val_path: datasets/tokenized/val.bin
  max_seq_len: 128
  num_workers: 0
  pin_memory: false
"""
with open("configs/pro_gpu.yaml", "w") as f:
    f.write(config)

# Fix 3 — patch trainer.py
with open("slm/training/trainer.py", "r") as f:
    t = f.read()
t = t.replace(
    "torch.cuda.amp.GradScaler(enabled=",
    "torch.amp.GradScaler('cuda', enabled="
)
with open("slm/training/trainer.py", "w") as f:
    f.write(t)

print("✅ All fixes applied!\n", flush=True)

# Step 4: Verify GPU
import torch
print(f"🖥️  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU!'}", flush=True)

print("\n✅ Setup complete! Now run the training cell below.", flush=True)

📥 Cloning project from GitHub...
Cloning into 'project'...
remote: Enumerating objects: 108, done.
remote: Counting objects: 100% (108/108), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 108 (delta 13), reused 100 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (108/108), 125.94 KiB | 969.00 KiB/s, done.
Resolving deltas: 100% (13/13), done.

📦 Installing dependencies...

🔧 Applying fixes...
✅ All fixes applied!

🖥️  GPU: Tesla T4

✅ Setup complete! Now run the training cell below.


In [3]:
import sys, os, time, math, torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

os.chdir('/content/project')
sys.path.insert(0, '/content/project')

from slm.config import EmilyConfig
from slm.tokenizer import EmilyTokenizer
from slm.dataset.loader import DatasetLoader
from slm.dataset.collator import CausalLMCollator
from slm.model import EmilySLM

cfg      = EmilyConfig.from_yaml("configs/pro_gpu.yaml")
tcfg     = cfg.training
tokenizer = EmilyTokenizer.load("checkpoints/emily-pro/tokenizer.json")
loader   = DatasetLoader(tokenizer, seq_len=128)
train_ds = loader.from_binary(cfg.dataset.train_path)
collator = CausalLMCollator(pad_token_id=0, max_length=128)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,
                          num_workers=0, collate_fn=collator, drop_last=False)

device = torch.device("cuda")
model  = EmilySLM(cfg.model).to(device)
print(f"Model: {sum(p.numel() for p in model.parameters()):,} params | "
      f"dataset: {len(train_ds)} samples | batches: {len(train_loader)}", flush=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1, betas=(0.9,0.95))
scaler    = torch.amp.GradScaler('cuda')
os.makedirs("checkpoints/emily-pro/best", exist_ok=True)

def get_lr(step):
    if step < 500: return 3e-4 * step / 500
    p = (step - 500) / max(1, 10000 - 500)
    return 3e-5 + 0.5*(3e-4 - 3e-5)*(1 + math.cos(math.pi * p))

model.train()
train_iter  = iter(train_loader)
best_loss   = float("inf")
accum_loss  = 0.0
t0          = time.time()
optimizer.zero_grad()

for step in range(1, 10001):
    try:
        batch = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        batch = next(train_iter)

    input_ids = batch["input_ids"].to(device)
    labels    = batch["labels"].to(device)

    with torch.autocast(device_type="cuda", dtype=torch.float16):
        logits = model(input_ids)["logits"]
        loss   = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                 labels.view(-1), ignore_index=-100) / 4
    scaler.scale(loss).backward()
    accum_loss += loss.item()

    if step % 4 == 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        for pg in optimizer.param_groups: pg["lr"] = get_lr(step)
        scaler.step(optimizer); scaler.update(); optimizer.zero_grad()

    if step % 50 == 0:
        t1 = time.time()
        tps = 50 * 8 * 128 / (t1 - t0)
        print(f"step={step:>6,}  loss={accum_loss*4:.4f}  lr={get_lr(step):.2e}  tok/s={tps:,.0f}", flush=True)
        accum_loss = 0.0; t0 = time.time()

    if step % 1000 == 0:
        torch.save(model.state_dict(), f"checkpoints/emily-pro/best/model.pt")
        print(f"  💾 Checkpoint saved at step {step}", flush=True)

torch.save(model.state_dict(), "checkpoints/emily-pro/best/model.pt")
print("\n🎉 Training complete!", flush=True)

FileNotFoundError: Tokenizer file not found: checkpoints/emily-pro/tokenizer.json

In [ ]:
import sys, os, time, math, torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

os.chdir('/content/project')
sys.path.insert(0, '/content/project')

# ── STEP 1: Prepare (build tokenizer + dataset) ───────────────────
print("="*55, flush=True)
print(" STEP 1: Preparing tokenizer and dataset", flush=True)
print("="*55, flush=True)

from slm.config import EmilyConfig
from slm.tokenizer import EmilyTokenizer
from slm.dataset.preprocessor import TextPreprocessor
from slm.dataset.loader import DatasetLoader
from slm.dataset.collator import CausalLMCollator
from slm.model import EmilySLM

cfg  = EmilyConfig.from_yaml("configs/pro_gpu.yaml")

# Write corpus
corpus_path = "data/raw/corpus.txt"
os.makedirs("data/raw", exist_ok=True)
raw = open(corpus_path).read() if os.path.exists(corpus_path) else ""
if not raw:
    # Pull from bootstrap.py corpus
    import importlib.util
    spec = importlib.util.spec_from_file_location("bootstrap", "bootstrap.py")
    bs   = importlib.util.load_from_spec(spec)
    spec.loader.exec_module(bs)
    raw  = bs.CORPUS
    open(corpus_path, "w").write(raw)

paragraphs = [p.strip() for p in raw.split("\n") if p.strip()]
pp    = TextPreprocessor(min_length=5, dedup=False)
texts = pp.process(paragraphs)
print(f"  Segments: {len(texts):,}", flush=True)

# Train tokenizer
os.makedirs("checkpoints/emily-pro", exist_ok=True)
tok_path = "checkpoints/emily-pro/tokenizer.json"
print("  Training BPE tokenizer...", flush=True)
tokenizer = EmilyTokenizer.train(texts, vocab_size=4096, show_progress=True)
tokenizer.save(tok_path)
print(f"  Tokenizer saved! vocab={len(tokenizer)}", flush=True)

# Build binary dataset
loader = DatasetLoader(tokenizer, seq_len=128)
train_p, val_p = DatasetLoader.tokenise_and_save(
    texts, tokenizer,
    output_path="datasets/tokenized",
    val_ratio=0.1
)
train_ds = loader.from_binary(train_p)
print(f"  Dataset ready! train samples={len(train_ds)}", flush=True)

# ── STEP 2: Train ─────────────────────────────────────────────────
print("\n" + "="*55, flush=True)
print(" STEP 2: Training on GPU", flush=True)
print("="*55 + "\n", flush=True)

collator     = CausalLMCollator(pad_token_id=0, max_length=128)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,
                          num_workers=0, collate_fn=collator, drop_last=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = EmilySLM(cfg.model).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"  Device : {device}", flush=True)
print(f"  Params : {n_params:,}", flush=True)
print(f"  Samples: {len(train_ds)}", flush=True)
print(f"  Batches: {len(train_loader)}", flush=True)
print(flush=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4,
                               weight_decay=0.1, betas=(0.9, 0.95))
scaler    = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None
os.makedirs("checkpoints/emily-pro/best", exist_ok=True)

def get_lr(step, warmup=500, max_steps=10000, max_lr=3e-4, min_lr=3e-5):
    if step < warmup:
        return max_lr * step / warmup
    p = (step - warmup) / max(1, max_steps - warmup)
    return min_lr + 0.5*(max_lr - min_lr)*(1 + math.cos(math.pi * p))

model.train()
train_iter = iter(train_loader)
best_loss  = float("inf")
run_loss   = 0.0
t0         = time.time()
optimizer.zero_grad()

for step in range(1, 10001):
    try:
        batch = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        batch = next(train_iter)

    ids    = batch["input_ids"].to(device)
    labels = batch["labels"].to(device)

    if scaler:
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            loss = F.cross_entropy(
                model(ids)["logits"].view(-1, cfg.model.vocab_size),
                labels.view(-1), ignore_index=-100) / 4
        scaler.scale(loss).backward()
    else:
        loss = F.cross_entropy(
            model(ids)["logits"].view(-1, cfg.model.vocab_size),
            labels.view(-1), ignore_index=-100) / 4
        loss.backward()

    run_loss += loss.item()

    if step % 4 == 0:
        if scaler:
            scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        for pg in optimizer.param_groups:
            pg["lr"] = get_lr(step)
        if scaler:
            scaler.step(optimizer); scaler.update()
        else:
            optimizer.step()
        optimizer.zero_grad()

    if step % 50 == 0:
        elapsed = time.time() - t0
        tps = 50 * 8 * 128 / elapsed
        print(f"step={step:>6,}  loss={run_loss*4:.4f}  "
              f"lr={get_lr(step):.2e}  tok/s={tps:,.0f}", flush=True)
        run_loss = 0.0
        t0 = time.time()

    if step % 1000 == 0:
        torch.save(model.state_dict(), "checkpoints/emily-pro/best/model.pt")
        print(f"  💾 Checkpoint saved at step {step}", flush=True)

torch.save(model.state_dict(), "checkpoints/emily-pro/best/model.pt")
print("\n🎉 Training complete!", flush=True)

 STEP 1: Preparing tokenizer and dataset
  Segments: 4,020
  Training BPE tokenizer...
  Tokenizer saved! vocab=2590
  Dataset ready! train samples=623

 STEP 2: Training on GPU

  Device : cuda
  Params : 35,660,288
  Samples: 623
  Batches: 78

step=    50  loss=504.8644  lr=3.00e-05  tok/s=17,403
step=   100  loss=406.8184  lr=6.00e-05  tok/s=26,715
step=   150  loss=328.4408  lr=9.00e-05  tok/s=24,093
step=   200  loss=284.9907  lr=1.20e-04  tok/s=21,806
step=   250  loss=242.4066  lr=1.50e-04  tok/s=26,918
step=   300  loss=195.3748  lr=1.80e-04  tok/s=26,759
step=   350  loss=154.6627  lr=2.10e-04  tok/s=26,827
step=   400  loss=126.7320  lr=2.40e-04  tok/s=26,786
step=   450  loss=106.2323  lr=2.70e-04  tok/s=25,135
step=   500  loss=87.9928  lr=3.00e-04  tok/s=21,233
step=   550  loss=68.9592  lr=3.00e-04  tok/s=26,660
step=   600  loss=45.8955  lr=3.00e-04  tok/s=26,153
step=   650  loss=25.2093  lr=3.00e-04  tok/s=26,369
step=   700  loss=11.8442  lr=3.00e-04  tok/s=26,014
st